In [104]:
import pymupdf
import spacy
import re
import pandas as pd
import numpy as np
import unicodedata
import os
from pathlib import Path

In [62]:
nlp = spacy.load('en_core_web_lg')

In [63]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances', 'housing']

In [55]:
# prod code

def find_uop(document, language):

    pdf = pymupdf.open(document)

    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria']

    elif language == 'PT':
        keywordsUOP = ['Uso de Recursos']
        keywordsSEEGP = ['Processo de Avaliação e Seleção de Projetos']

    elif language == 'ES':
        keywordsUOP = ['Uso de fondos', 'Uso de los fondos']
        keywordsSEEGP = ['XYZ']

    areaUOP = None
    areaSEEGP = None

    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

        if areaUOP is None:
            for keyword in keywordsUOP:
                start = page.search_for(keyword)
                if start:
                    areaUOP = (page_idx, start[0])

        if areaSEEGP is None:
            for keyword in keywordsSEEGP:
                end = page.search_for(keyword)
                if end:
                    areaSEEGP = (page_idx, end[0])

    return areaUOP, areaSEEGP

In [23]:
# iterate pages in the pdf and extract preferrably tables or else words from UOP section

def page_scenario_and_extract(document, areaUOP, areaSEEGP):

    pdf = pymupdf.open(document)
    # language = language

    # inputs
    start_page_idx = areaUOP[0]
    end_page_idx = areaSEEGP[0]
    start_point = areaUOP[1].y1
    end_point = areaSEEGP[1].y0

    # outputs
    noTableMsg = []
    hasTableMsg = []
    hasDFMsg = []
    extractUOPwords = []

# iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

# process four page scenarios, check for tables, extract tables else extract text as words
    # A: UOP all on a single page
        if page_idx == start_page_idx and page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point and bbox[3] < end_point:
                    tableAheader = tables[0].header.names
                    tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
            else:
                noTableMsg.append('No tableA')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point and y1 < end_point:
                        extractUOPwords.append(text)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == start_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point:
                    tableBheader = tables[0].header.names
                    tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
            else:
                noTableMsg.append('No tableB')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point:
                        extractUOPwords.append(text)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > start_page_idx and page_idx < end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                tableDheader = tables[0].header.names
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
            else:
                noTableMsg.append(page_idx)
                noTableMsg.append('No tableD')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    extractUOPwords.append(text)

    # C: current page is end page
        elif page_idx == end_page_idx:
            tables = page.find_tables()
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[3] < end_point:
                    tableCheader = tables[0].header.names
                    tableCdf = tables[0].to_pandas()
                hasTableMsg.append(tableCheader)
                hasDFMsg.append(tableCdf)
            else:
                noTableMsg.append('No tableC')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y1 < end_point:
                        extractUOPwords.append(text)

    return noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords

    # noTableMsg lists page scenarios (and idx for scenario D) where no table is found
    # hasTableMsg lists the header row for found tables
    # hasDFMsg dataframe content in a list

In [48]:
# check extracted table is UOP table and extract Categories

def UOP_table_cats(hasDFMsg, hasTableMsg):

    # inputs
    hasDFMsg = hasDFMsg
    hasTableMsg = hasTableMsg

    # outputs
    uniqueCats = []
    tableInfo = []

    if len(hasDFMsg) > 1:
        tableInfo.append('More than one table found')
    if len(hasDFMsg) == 0:
        tableInfo.append('No table found')
    
    if len(hasDFMsg) == 1:
    # this over simplifies by assuming the category is always in the first column
    # the header condition code oversimplifies by assuming table fragments split across pages without a header row contain no new category labels

        if 'Category' in hasTableMsg[0]:
            x = 'singular'
        elif 'Categories' in hasTableMsg[0]:
            x = 'plural'
        elif 'Eligible Project Category' in hasTableMsg[0]:
            x = 'phrase'
        else:
            x = 'not a UOP table or category not in 1st columns'

        DFname = hasDFMsg[0]
        if x == 'singular':
            uniqueC = DFname['Category'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'plural':
            uniqueC = DFname['Categories'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'phrase':
            uniqueC = DFname['Eligible Project Category'].unique()
            uniqueCats.append(uniqueC)
        else:
            tableInfo.append(x)

    return tableInfo, uniqueCats

    # turn these into assert and proper error msgs later
        # if errorMsg is empty and uniqueCats contains a list of category like words, pdf has processed successfully
        # if errorMsg is not empty, there may be more than one table found, in which case DFname variable could be inaccurate
        # if errorMsg is not empty, the words Category or Categories may not be present in the header row, in which case may not be a UOP table or may be other words such as criteria


In [199]:
def catTable(language, document):

    if language == 'EN':
        keywordsCAT = ['Category', 'Categories', 'Criteria', 'Criterion']
    
    pdf = pymupdf.open(document) 

    tableCount = 0
    tablePages = []
    tableHeaders = []
    tableDFs = []


    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]

        find_tables = page.find_tables(strategy = 'lines_strict')
        if find_tables.tables:
            table_count = len(find_tables.tables)
            tableheader = find_tables[0].header.names # for now assume only 1 table per page
            tableDF = find_tables[0].to_pandas()
            tableCount += table_count
            tablePages.append(page_idx)
            tableHeaders.append(tableheader)
            tableDFs.append(tableDF)

    return tableCount, tablePages, tableHeaders, tableDFs

        

In [58]:
def fontInfo(language, document):
    
    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']
        keywordsSEEGP_test = ['Selection and Evaluation', 'Process for', 'Evaluation and Selection', 'Project Evaluation', 'Assessment Process', 'Selection process', 'Project selection criteria']


    results_UOP = []
    results_SEEGP = []
    
    pdf = pymupdf.open(document) 

    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]
        dict = page.get_text("dict")
        blocks = dict["blocks"] 
        for block in blocks:
            if "lines" in block.keys():
                spans = block['lines']
                for span in spans:
                    data = span['spans']
                    for lines in data:
                        for keyword in keywordsUOP:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_UOP.append((lines['text'], lines['size'], lines['bbox'], page_idx))
                        for keyword in keywordsSEEGP_test:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_SEEGP.append((lines['text'], lines['size'], lines['bbox'], page_idx))

                            

    return results_UOP, results_SEEGP

In [167]:
def find_max_font(results_UOP, results_SEEGP):
    
    max_font_size = 0
    for result_idx in range(len(results_UOP)):
        result = results_UOP[result_idx]
        if result[1] > max_font_size:
            max_font_size = result[1]
            max_idx_UOP = result_idx

    max_font_size = 0
    if len(results_SEEGP) == 0:
        max_idx_SEEGP = 0
    else:       
        for result_idx in range(len(results_SEEGP)):
            result = results_SEEGP[result_idx]
            if result[1] > max_font_size:
                max_font_size = result[1]
                max_idx_SEEGP = result_idx

    return max_idx_UOP, max_idx_SEEGP

In [172]:
def tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP):
    
    if len(results_SEEGP) == 0:
        areaUOP = results_UOP[max_idx_UOP]
        areaUOP_page = areaUOP[3]
        areaUOP_coords = areaUOP[2]
        areaUOP_y1 = areaUOP_coords[3]
        areaSEEGP_page = 0
        areaSEEGP_y0 = 0

    else:
        areaUOP = results_UOP[max_idx_UOP]
        areaSEEGP = results_SEEGP[max_idx_SEEGP]
        areaUOP_page = areaUOP[3]
        areaSEEGP_page = areaSEEGP[3]
        areaUOP_coords = areaUOP[2]
        areaSEEGP_coords = areaSEEGP[2]
        areaUOP_y1 = areaUOP_coords[3]
        areaSEEGP_y0 = areaSEEGP_coords[1]

        if areaUOP_page == areaSEEGP_page:
            if 0 < areaSEEGP_y0 - areaUOP_y1 < 40:
                results_UOP_remove = results_UOP.pop(max_idx_UOP)
                results_SEEGP_remove = results_SEEGP.pop(max_idx_SEEGP)

                max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)

                areaUOP = results_UOP[max_idx_UOP]
                areaSEEGP = results_SEEGP[max_idx_SEEGP]
                areaUOP_page = areaUOP[3]
                areaSEEGP_page = areaSEEGP[3]
                areaUOP_coords = areaUOP[2]
                areaSEEGP_coords = areaSEEGP[2]
                areaUOP_y1 = areaUOP_coords[3]
                areaSEEGP_y0 = areaSEEGP_coords[1]

    return areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0

In [61]:
def keepSEEGP(areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0):

    if areaSEEGP_page == areaUOP_page:
        if areaSEEGP_y0 < areaUOP_y1:
            keep_areaSEEGP = False
        elif areaSEEGP_y0 > areaUOP_y1:
            keep_areaSEEGP = True
    elif areaSEEGP_page > areaUOP_page:
        keep_areaSEEGP = True
    elif areaSEEGP_page < areaUOP_page:
        keep_areaSEEGP = False

    return keep_areaSEEGP

In [142]:
def tableAssess(language, tableCount, tablePages, tableHeaders, tableDFs):

    if language == 'EN':
        keywordsCAT = ['Category', 'Categories', 'Criteria', 'Criterion', 'Green Sectors']

    uniqueCats = []
    uniqueCatsNoDupes = []
    tablePageCurrent = 0
    tablePagePrior = 0

    if tableCount == 1: # applicable if only 1 table detected in whole pdf
        dataFrame = tableDFs[0]
        header = tableHeaders[0]
        for keyword in keywordsCAT:
            if keyword.lower() in header[0].lower(): # assumes category col is always the first (LHS) col
                colName = header[0]
                uniqueC = dataFrame[colName].dropna().unique().tolist()
                for messyCat in uniqueC:
                    if messyCat:
                        tidyCat = messyCat.replace('\n',' ').lower()
                        uniqueCatsNoDupes.append(tidyCat)
    elif tableCount > 1:
        for idx in range(len(tableDFs)): # count of tableHeaders and DFs should be the same, based on the page level find_tables object
            tablePageCurrent = tablePages[idx]
            dataFrame = tableDFs[idx]
            header = tableHeaders[idx]
            for keyword in keywordsCAT:
                if keyword.lower() in header[0].lower():
                    colName = header[0]
                    uniqueC = dataFrame[colName].dropna().unique().tolist()
                    for messyCat in uniqueC:
                        if messyCat:
                            tidyCat = messyCat.replace('\n',' ').lower()
                            uniqueCats.append(tidyCat)
                elif header[0] == 'Col0' and tablePageCurrent - tablePagePrior == 1:
                    uniqueC = dataFrame['Col0'].dropna().unique().tolist()
                    if uniqueC:
                        for messyCat in uniqueC:
                            if messyCat:
                                tidyCat = messyCat.replace('\n',' ').lower()
                                uniqueCats.append(tidyCat)
            tablePagePrior = tablePageCurrent
        uniqueCatsNoDupes = list(set(uniqueCats))

    return uniqueCatsNoDupes

# good example of use of AI: Gemini: https://www.google.com/search?client=safari&rls=en&q=I+just+want+the+unique+categories+of+a+dataframe+column+returned+by+unique%28%29+so+how+do+I+discard+the+other+info+that+is+returned%3F+the+unique+values+are+in+a+list+like+structure+inside+the+numpy+array&ie=UTF-8&oe=UTF-8
# 
        

In [201]:
document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/Green_Finance_Framework_Interchile.pdf'
language = 'EN'

tableCount, tablePages, tableHeaders, tableDFs = catTable(language, document)
uniqueCats = tableAssess(language, tableCount, tablePages, tableHeaders, tableDFs)
print(tableCount)
print(tablePages)
# print(tableHeaders)
print(uniqueCats)

results_UOP, results_SEEGP = fontInfo(language, document)
max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)
areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0 = tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP)
keep_areaSEEGP = keepSEEGP(areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0)

print(areaUOP_page, areaUOP_y1)
print(areaSEEGP_page, areaSEEGP_y0)
print(keep_areaSEEGP)





1
[8]
['renewable energy', 'energy efficiency', 'climate change adaptation']
7 321.5299987792969
9 111.74002075195312
True


In [79]:
# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(extractUOPwords):

    # inputs
    extractUOPwords = extractUOPwords

    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}

    #doca = ''
    #docc = ''
    #docd = ''
    #doce = ''
    #docf = ''
    #docg = ''
    #doch = ''
    #doci = ''
    #docj = ''
    #dock = ''

    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim
                #dfre = pd.DataFrame.from_dict(simsre, 'index')
            #else:
                #dfre = 'is not re'


    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
                #dfee = pd.DataFrame.from_dict(simsee, 'index')
            #else:
                #dfee = 'is not ee'
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim
                #dfppc = pd.DataFrame.from_dict(simsppc, 'index')
            #else:
                #dfppc = 'is not ppc'

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim
                #dfesml = pd.DataFrame.from_dict(simsesml, 'index')
            #else:
                #dfesml = 'is not esml'

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim
                #dftabc = pd.DataFrame.from_dict(simstabc, 'index')
            #else:
                #dftabc = 'is not tabc'

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim
                #dfct = pd.DataFrame.from_dict(simsct, 'index')
            #else:
                #dfct = 'is not ct'

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim
                #dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')
            #else:
                #dfswwm = 'is not swwm'

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim
                #dfcca = pd.DataFrame.from_dict(simscca, 'index')
            #else:
                #dfcca = 'is not cca'

    # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim
                #dfce = pd.DataFrame.from_dict(simsce, 'index')
            #else:
                #dfce = 'is not ce'

    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        docb = ''
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim
                #dfgb = pd.DataFrame.from_dict(simsgb, 'index')
            #else:
                #dfgb = 'is not gb'

    return simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb

In [80]:
# run the program
# notable



document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/CADU_GREEN_BOND_FRAMEWORK.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)
noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords = page_scenario_and_extract(document, areaUOP, areaSEEGP)
tableInfo, uniqueCats = UOP_table_cats(hasDFMsg, hasTableMsg)
if len(uniqueCats) == 0:
    simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb = UOPwords_to_Catwords(extractUOPwords)
filepath = Path(document)

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/1943418420.py:50: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/1943418420.py:64: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/1943418420.py:77: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/1943418420.py:90: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doce.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/1943418420.py:103: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docf.similarity(docb)

In [34]:
# run the program
# notable

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/CADU_GREEN_BOND_FRAMEWORK.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)
noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords = page_scenario_and_extract(document, areaUOP, areaSEEGP)
if len(hasDFMsg) == 0:
    dfre, dfee, dfppc, dfesml, dftabc, dfct, dfswwm, dfcca, dfce, dfgb = UOPwords_to_Catwords(extractUOPwords)
elif len(hasDFMsg) != 0:
    tableInfo, uniqueCats = UOP_table_cats(hasDFMsg, hasTableMsg)
    if len(uniqueCats) == 0:
        dfre, dfee, dfppc, dfesml, dftabc, dfct, dfswwm, dfcca, dfce, dfgb = UOPwords_to_Catwords(extractUOPwords)
filepath = Path(document)

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/3845112704.py:39: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/3845112704.py:52: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/3845112704.py:64: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/3845112704.py:76: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doce.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_97458/3845112704.py:88: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docf.similarity(docb) 

In [81]:
print(areaUOP)
print(areaSEEGP)
print(extractUOPwords)
print(len(extractUOPwords))
print(noTableMsg)
print(hasTableMsg)
print(hasDFMsg)
print(simsgb)




# print(hasTableMsg[0])
# print(len(extractUOPwords))

(6, Rect(84.98400115966797, 127.81880187988281, 171.4713592529297, 140.12840270996094))
(8, Rect(84.98400115966797, 308.5887451171875, 318.9053955078125, 320.8983459472656))
['6', 'Proceeds', 'from', 'CADU’s', 'green', 'bonds', 'will', 'be', 'used', 'to', 'finance', 'or', 'refinance', 'the', 'Company’s', 'existing4', 'or', 'future', 'housing', 'projects', 'that', 'are', 'aligned', 'with', 'an', 'environment-', 'friendly', 'framework', '(“Green', 'Projects”).', 'That', 'is,', 'that', 'these', 'projects', 'should', 'already', 'have', 'the', 'ECOCOSA,', 'the', 'EDGE', 'or', 'any', 'other', 'environmental', 'certifications', 'approved', 'by', 'the', 'CBI.', 'The', 'intention', 'is', 'to', 'be', 'in', 'line', 'with', 'the', '“Low', 'Carbon', 'Buildings', 'Criteria', 'for', 'Residential', 'Buildings”.', 'Green', 'Projects', 'should', 'also', 'be', 'in', 'line', 'with', 'the', 'Sustainable', 'Development', 'Goal', '(SDG)', '11', 'which', 'aims', 'to', 'make', 'cities', 'and', 'human', 'settle

In [85]:
# collate the outputs
# notable

print(filepath.name, type(filepath.name))
print(language, type(language))
print(areaUOP[0], type(areaUOP[0]))
print(areaSEEGP[0], type(areaSEEGP[0]))
if len(hasDFMsg) != 0:
    if len(tableInfo) == 1:   
        print(tableInfo[0], type(tableInfo[0]))
    if len(tableInfo) == 2:
        print(tableInfo[1], type(tableInfo[1]))
    if len(uniqueCats) > 0:
        print(uniqueCats, len(uniqueCats))
    else:
        print('no categories from tables')
if len(hasDFMsg) == 0:
    print(f"renewable_energy  {simsre}")
    print(f"energy_efficiency {simsee}")
    print(f"pollution_prevention_and_control {simsppc}")
    print(f"environmentally_sustainable_management_of_living_natural_resources_and_land_use {simsesml}")
    print(f"terrestrial_and_aquatic_biodiversity_conservation {simstabc}")
    print(f"clean_transportation {simsct}")
    print(f"sustainable_water_and_wastewater_management {simsswwm}")
    print(f"climate_change_adaptation {simscca}")
    print(f"circular_economy_and_or_ecoefficient_projects {simsce}")
    print(f"green_buildings {simsgb}")
else:
    print('categories from tables')

CADU_GREEN_BOND_FRAMEWORK.pdf <class 'str'>
EN <class 'str'>
6 <class 'int'>
8 <class 'int'>
renewable_energy  {}
energy_efficiency {'efficiency efficiency': 1.0}
pollution_prevention_and_control {'pollution environmental': 0.7345117926597595, 'pollution emissions': 0.7256803512573242, 'waste waste': 1.0}
environmentally_sustainable_management_of_living_natural_resources_and_land_use {}
terrestrial_and_aquatic_biodiversity_conservation {}
clean_transportation {}
sustainable_water_and_wastewater_management {'water water': 0.8583021759986877, 'wastewater sewage': 0.8658150434494019}
climate_change_adaptation {}
circular_economy_and_or_ecoefficient_projects {}
green_buildings {'buildings Buildings': 0.7504333257675171, 'buildings houses': 0.7476946711540222, 'housing housing': 1.0, 'housing Housing': 1.0}


In [21]:
print(len(noTableMsg))

12


In [22]:
start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0

print(start_page_idx)
print(end_page_idx)
print(start_point)
print(end_point)

6
6
140.12840270996094
422.3687744140625


In [123]:
import string

extractUOPwords_cleaned = []
remove_punct = str.maketrans('','',string.punctuation)
for word in extractUOPwords:
    cleaned_word = word.translate(remove_punct).strip()
    if cleaned_word:
        extractUOPwords_cleaned.append(cleaned_word)
print(extractUOPwords_cleaned)
print(type(extractUOPwords_cleaned))


['12', 'The', 'proceeds', 'to', 'be', 'raised', 'through', 'green', 'bond', 'issuances', 'will', 'be', 'used', 'to', 'finance', 'projects', 'in', 'the', 'following', 'categories', 'Bioenergy12', 'Projects', 'related', 'to', 'production', 'of', 'hydrous', 'and', 'anhydrous', 'cornethanol', 'biofuel', 'including', 'i', 'capital', 'expenditures', 'for', 'development', 'construction', 'operation', 'and', 'maintenance', 'of', 'biofuel', 'production', 'facilities', 'or', 'ii', 'operational', 'expenditures', 'or', 'refinance', 'of', 'purchased', 'corn', 'feedstock', 'for', 'biofuel', 'production', 'Feedstock', 'will', 'be', 'purchased', 'from', 'suppliers', 'in', 'compliance', 'with', 'FS', 'Sustainability', 'Protocol', 'andor', 'certified', 'against', 'the', 'Climate', 'Bonds', 'Standard', 'Agriculture', 'Criteria', 'Version', '1', '1', 'Excludes', 'fossil', 'biofuel', 'production', 'and', 'blending', 'facilities', '2', 'The', 'hydrated', 'ethanol', 'is', 'the', 'common', 'ethanol', 'sold', 

In [69]:
# run the program
# X - non UOP table, 100% ES

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)
noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords = page_scenario_and_extract(document, areaUOP, areaSEEGP)
# tableInfo, uniqueCats = UOP_table_cats(hasDFMsg, hasTableMsg)
# if len(uniqueCats) == 0:
    # dfre, dfee, dfppc, dfesml, dftabc, dfct, dfswwm, dfcca, dfce, dfgb = UOPwords_to_Catwords(extractUOPwords)
# filepath = Path(document)

In [70]:
print(areaUOP)
print(areaSEEGP)
print(extractUOPwords)

(1, Rect(85.10399627685547, 407.7648620605469, 171.381591796875, 420.328369140625))
(2, Rect(85.10399627685547, 354.48486328125, 215.33181762695312, 367.0483703613281))
['Guidance', 'for', 'Climate', 'Bond', 'Framework', 'as', 'a', 'requirement', 'for', 'CBI', 'For', 'future', 'expenses,', 'with', '40', 'projects', 'not', 'yet', 'defined,', 'the', 'process', 'of', 'selecting', 'projects', 'for', 'financing', 'begins', 'with', 'the', "client's", 'expression', 'of', 'interest', 'in', 'self-sufficiency', 'in', 'the', 'production', 'and', 'compensation', 'of', 'electricity,', 'and', 'also', 'includes', 'approval', 'of', 'the', 'local', 'distribution', 'concessionaire,', 'according', 'to', 'the', 'process', 'described', 'in', 'the', 'following', 'item.', 'GBP', 'explicitly', 'recognizes', 'Renewable', 'Energy', 'as', 'a', 'project', 'category', 'eligible', 'for', 'characterization', 'as', 'Green', 'Bond.', 'Additionally,', 'the', 'project', 'category', 'related', 'to', 'solar', 'energy', 'i

In [73]:
# inputs
extractUOPwords = extractUOPwords

re = renewable_energy
retest = ['biofuel', 'forestry', 'forest', 'housing']
ee = energy_efficiency
ppc = pollution_prevention_and_control
esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
tabc = terrestrial_and_aquatic_biodiversity_conservation
ct = clean_transportation
swwm = sustainable_water_and_wastewater_management
cca = climate_change_adaptation
ce = circular_economy_and_or_ecoefficient_projects
gb = green_buildings

similarity_threshold = 0.72

# outputs
simsre = {}
simsee = {}
simsppc = {}
simsesml = {}
simstabc = {}
simsct = {}
simsswwm = {}
simscca = {}
simsce = {}
simsgb = {}

extractUOPTest = ['biofuel', 'dog', 'cat', 'lion', 'forestry', 'forest', 'housing']

doca = ''
doce = ''
dock = ''

# renewable_energy
for worda in re:
    doca = nlp(worda)
    docb = ''
    for wordb in extractUOPwords:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim
            dfre = pd.DataFrame.from_dict(simsre, 'index')
        else:
            dfre = 'is not re'

 # environmentally_sustainable_management_of_living_natural_resources_and_land_use
for worde in retest:
    doce = nlp(worde)
    docb = ''
    for wordb in extractUOPTest:
        docb = nlp(wordb)
        if doce.similarity(docb) >= similarity_threshold:
            sim = doce.similarity(docb)
            simsesml[doce[0].text + ' ' + docb[0].text] = sim
            dfesml = pd.DataFrame.from_dict(simsesml, 'index')
        else:
            dfesml = 'is not esml'

# green_buildings
for wordk in retest:
    dock = nlp(wordk)
    docb = ''
    for wordb in extractUOPTest:
        docb = nlp(wordb)
        if dock.similarity(docb) >= similarity_threshold:
            sim = dock.similarity(docb)
            simsgb[dock[0].text + ' ' + docb[0].text] = sim
            dfgb = pd.DataFrame.from_dict(simsgb, 'index')
        else:
            dfgb = 'is not gb'

            

In [74]:
print(dfre)
print(dfesml)
print(dfgb)

                                  0
renewable Renewable        1.000000
renewable renewable        1.000000
solar solar                1.000000
solar photovoltaic         0.790688
transmission Transmission  1.000000
generation generation      0.822127
                     0
biofuel biofuel    1.0
forestry forestry  1.0
forest forest      1.0
housing housing    1.0
                     0
biofuel biofuel    1.0
forestry forestry  1.0
forest forest      1.0
housing housing    1.0


In [86]:
x = 'biofuel'
y = 'biofuel'

xx = nlp(x)
yy = nlp(y)

xx.similarity(yy)


1.0

In [89]:
try:
    index = extractUOPwords.index('biofuel')
    print(index)

except ValueError:
    print('not found')

42


very strange
the dictionaries are working but the df's , except for re, are not.

construct the dfs outside the funciton as that seems to work

strip the punct from the FS July 2021 extract words list

introduce both changes to the main code and re-test

you've fixed no table , found 3 other improvements: 1) pagination, dict not df, clean punct

go onto FS Nov 2020

then turn attn to the pagination soln for the UOP area